# Week 21: High-Performance Inference & Compilers
## Notebook 07: vLLM Chunked Prefill & Mixed Batching Scheduler

### 1. Conceptual Intuition & Theoretical Background
In production LLM serving, user requests arrive continuously with two distinct phases:
1. **Prefill Phase:** Computes KV caches for input prompts ($Q \gg 1$). This phase is **compute-bound**.
2. **Decode Phase:** Generates one token per step for active streams ($Q = 1$). This phase is **memory-bandwidth bound**.

**The Prefill Bubble Problem:**
When a long prompt arrives (e.g. 8,000 tokens), traditional schedulers pause all active decode requests to run the prefill forward pass. This causes significant latency spikes:
$$\text{Inter-Token Latency (ITL) Spikes} \gg 500\text{ms}$$

**Chunked Prefill (Sarathi-Serve / vLLM):**
Partitions long prompt prefills into smaller chunks of size $C$ (e.g. $C = 512$).
In each iteration, the scheduler batches:
- Decode tokens from active generation requests.
- A **chunk** of a pending prefill request.

$$\text{Iteration Token Budget } T_{\text{budget}} = C_{\text{prefill}} + B_{\text{decode}}$$
This saturates GPU compute cores while keeping inter-token generation latency smooth and stable.

---

### 2. Scratch Python Implementation: Chunked Prefill Scheduler

In [ ]:
class Request:
    def __init__(self, req_id, prompt_tokens, is_decode=False):
        self.id = req_id
        self.remaining_prompt = prompt_tokens
        self.generated_tokens = 0
        self.is_decode = is_decode

class ChunkedPrefillScheduler:
    def __init__(self, token_budget=512):
        self.token_budget = token_budget
        self.waiting_queue = []
        self.running_decodes = []
        
    def add_request(self, req):
        self.waiting_queue.append(req)
        
    def schedule_iteration(self):
        tokens_budget_left = self.token_budget
        scheduled_decodes = []
        scheduled_prefill_chunks = []
        
        # 1. Prioritize active decode requests to guarantee ITL SLA
        for req in self.running_decodes:
            if tokens_budget_left >= 1:
                scheduled_decodes.append(req.id)
                tokens_budget_left -= 1
                req.generated_tokens += 1
                
        # 2. Fill remaining GPU capacity with prompt prefill chunks
        if self.waiting_queue and tokens_budget_left > 0:
            current_prefill = self.waiting_queue[0]
            chunk_tokens = min(current_prefill.remaining_prompt, tokens_budget_left)
            current_prefill.remaining_prompt -= chunk_tokens
            tokens_budget_left -= chunk_tokens
            
            scheduled_prefill_chunks.append((current_prefill.id, chunk_tokens))
            
            # If prefill completed, promote request to running decode queue
            if current_prefill.remaining_prompt == 0:
                current_prefill.is_decode = True
                self.running_decodes.append(self.waiting_queue.pop(0))
                
        return scheduled_decodes, scheduled_prefill_chunks, self.token_budget - tokens_budget_left

# Simulate High-Load Serving Engine
scheduler = ChunkedPrefillScheduler(token_budget=512)

# Existing active decode streams
for i in range(12):
    scheduler.running_decodes.append(Request(f"decode_{i}", 0, is_decode=True))

# Arrival of a long 1200-token prompt
scheduler.add_request(Request("long_doc_query", 1200))

print("--- Iterative Chunked Prefill Execution ---")
for step in range(4):
    decodes, prefills, total_tokens = scheduler.schedule_iteration()
    print(f"Step {step+1}: Scheduled {len(decodes)} Decodes + Prefill Chunks {prefills} | Total Tokens: {total_tokens}/512")

### 3. Step-by-Step Code Walkthrough
1. **Decode SLA Guarantee**: Allocates slots for active generation tokens first to prevent ITL stutter.
2. **Chunk Partitioning**: Slices long prompt prefills into 512-token segments.
3. **Compute Saturation**: Blends compute-bound prefill with memory-bound decode for high GPU utilization.

---

### 4. Industry Applications & Trade-offs
- **High-Throughput Production LLM Serving (vLLM, TensorRT-LLM):** Delivers sub-30ms inter-token latencies under heavy load.
- **Trade-off:** Attention kernels must support mixed sequence lengths within the same batch.

---

### 5. Top Interview Questions & Answers
**Q1: What is the "Prefill Bubble" and how does Chunked Prefill eliminate it?**
> **Answer:** Large prompt prefills monopolize the GPU and block active decoding streams, causing spikes in Inter-Token Latency (ITL). Chunked prefill splits large prompts into smaller token chunks that are co-scheduled alongside decode tokens, smoothing latency.

**Q2: Why does mixing Prefill and Decode tokens in the same batch improve overall GPU hardware utilization?**
> **Answer:** Prefill is compute-heavy (FLOPs bound) while decode is memory-bandwidth heavy (memory bound). Mixing them leverages both compute cores and memory bandwidth simultaneously.